# Alaga Alerta — treino do detector (YOLO) no Colab

Treina o detector de **rua alagada**, **bueiro entupido** e **veículo submerso** usando os mesmos scripts do repositório
(`scripts/prepare_dataset.py`, `scripts/train.py`, `scripts/evaluate.py`), então o resultado é reproduzível fora do notebook.

**Antes de rodar**
1. `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.
2. Crie uma conta grátis no [Roboflow](https://roboflow.com) e copie a API key (*Settings > API Keys*).
3. No Colab, abra o ícone de chave (🔑 *Secrets*) e crie o segredo `ROBOFLOW_API_KEY` com acesso liberado para este notebook.

Ao final, baixe `entrega_treino.zip` e copie `models/alaga_yolo.pt` para a pasta `models/` do projeto: é ele que a API carrega.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto

In [ ]:
# Opção A: clonar do GitHub (troque pela URL do repositório da equipe)
REPO_URL = "https://github.com/SEU_USUARIO/AlagaAlerta.git"
!git clone -q {REPO_URL} AlagaAlerta
%cd AlagaAlerta

# Opção B: enviar um .zip do projeto pelo painel de arquivos e descompactar:
# !unzip -q AlagaAlerta.zip -d /content && %cd /content/AlagaAlerta

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Opcional: as fontes principais (Kaggle e Wikimedia Commons) não precisam de login.
# Com uma ROBOFLOW_API_KEY nos Secrets do Colab, entram também os reforços do Roboflow.
import os
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    print("Roboflow: chave carregada")
except Exception:
    print("Roboflow: sem chave, fontes opcionais serão puladas")

## 2. Baixar e conferir as fontes do dataset

`coletar_bueiros.py` baixa do Wikimedia Commons as fotos de bueiro revisadas em `training/bueiros_commons.csv`.
Depois o `--inspecionar` baixa os datasets do Kaggle, gera os pseudo-rótulos de veículo submerso e mostra quantas
caixas de cada classe cada fonte gerou.

In [ ]:
!python scripts/coletar_bueiros.py
!python scripts/prepare_dataset.py --inspecionar

## 3. Montar o dataset unificado (train/val/test)

In [ ]:
!python scripts/prepare_dataset.py
from IPython.display import Markdown, display
display(Markdown(open("docs/dataset_stats.md", encoding="utf-8").read()))

## 4. Treinar

`yolo26s` é um bom equilíbrio para a T4 (~1 h para 100 épocas em alguns milhares de imagens).
Para rodar em CPU/borda depois, troque por `yolo26n.pt`. O early stopping (`--paciencia`) encerra se o mAP de validação parar de melhorar.

In [ ]:
!python scripts/train.py --modelo yolo26s.pt --epocas 100 --batch -1 --device 0

## 5. Avaliar no conjunto de TESTE (mAP, precisão, recall)

In [ ]:
!python scripts/evaluate.py --device 0 --exemplos 12
display(Markdown(open("docs/metricas.md", encoding="utf-8").read()))

In [ ]:
from IPython.display import Image
from pathlib import Path
for nome in ["treino_results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    p = Path("docs/figuras") / nome
    if p.exists():
        print(nome); display(Image(filename=str(p), width=900))
for p in sorted(Path("docs/figuras/exemplos").glob("*.jpg"))[:6]:
    display(Image(filename=str(p), width=500))

## 6. Baixar os artefatos para o repositório

In [ ]:
!zip -qr entrega_treino.zip models/alaga_yolo.pt models/alaga_yolo.json docs/metricas.md docs/metricas.json docs/dataset_stats.md docs/dataset_stats.json docs/figuras
from google.colab import files
files.download("entrega_treino.zip")

**Opcional:** o Colab apaga tudo ao desconectar. Para guardar os runs completos no Drive:

```python
from google.colab import drive; drive.mount("/content/drive")
!cp -r runs /content/drive/MyDrive/alaga_runs
```

Depois: descompacte o zip na raiz do projeto, faça commit de `docs/` e de `models/alaga_yolo.pt`.